In [1]:
# ── CELDA 0: Anti-desconexión (opcional, igual que antes) ──
import time, threading

def heartbeat():
    while True:
        time.sleep(45)
        try:
            from google.colab import output
            output.eval_js('document.querySelector("#top-toolbar").click()')
        except:
            pass

thread = threading.Thread(target=heartbeat, daemon=True)
thread.start()
print("✅ Anti-desconexión activo")

✅ Anti-desconexión activo


In [1]:
# ── CELDA 1: Verificar GPU
!nvidia-smi
import torch
print(f'\nCUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No se detectó GPU. El entrenamiento será lento en CPU.')

Mon Apr 20 23:24:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── CELDA 2: Instalar ultralytics (última versión con YOLO11) ──
!pip install ultralytics -q
from ultralytics import YOLO
print('✅ Ultralytics instalado (soporta YOLO11)')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics instalado (soporta YOLO11)


In [3]:
#CELDA 3: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

Mounted at /content/drive
✅ Google Drive montado en /content/drive


In [11]:
# ── CELDA 4: Verificar estructura de datasets ──────────────────
import os

DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'

datasets = {
    'global':   f'{DRIVE_BASE}/license-plates',
    'ecuador1': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1',
    'ecuador2': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2',
    'ecuador4': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4',
}

for name, path in datasets.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌'
    if exists:
        train_count = len(os.listdir(f'{path}/train/images')) if os.path.exists(f'{path}/train/images') else 0
        print(f'{status} {name}: {train_count} imágenes de entrenamiento')
    else:
        print(f'{status} {name}: NO ENCONTRADO en {path}')


✅ global: 7057 imágenes de entrenamiento
✅ ecuador1: 54 imágenes de entrenamiento
✅ ecuador2: 90 imágenes de entrenamiento
✅ ecuador4: 375 imágenes de entrenamiento


In [10]:
# ── CELDA 5: Crear data.yaml para combined_all (compatible YOLO11) ──
import yaml

data = {
    'train': [
        f'{DRIVE_BASE}/license-plates/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    'val':  f'{DRIVE_BASE}/license-plates/valid/images',
    'test': f'{DRIVE_BASE}/license-plates/test/images',
    'nc':   1,
    'names': ['license plate'],
}

yaml_path = '/content/data_combined_all.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('✅ data_combined_all.yaml creado')
print(f'   Train: {len(data["train"])} carpetas')
for p in data['train']:
    count = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'   - {p.split("/")[-3]}/{p.split("/")[-2]}: {count} imgs')


✅ data_combined_all.yaml creado
   Train: 4 carpetas
   - license-plates/train: 7057 imgs
   - license-plates-ec-1/train: 54 imgs
   - license-plates-ec-2/train: 90 imgs
   - license-plates-ec-4/train: 375 imgs


In [ ]:
# ── CELDA 6: Entrenar YOLO11n combinado (global + ecuador) ─────
# Usamos yolo11n.pt (nano)

from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # usar YOLO11

results = model.train(
    data     = '/content/data_combined_all.yaml',
    epochs   = 100,              # puedes mantener 100 o aumentarlo a 150 si quieres más precisión
    imgsz    = 640,
    batch    = 32,
    name     = 'yolo11n_combined_all',
    project  = '/content/drive/MyDrive/TrafficVision/runs',
    patience = 15,
    save     = True,
    plots    = True,
    device   = 0,                # GPU
    amp      = True,
    cos_lr   = True,
    warmup_epochs = 3,
    label_smoothing = 0.0,
    dropout = 0.0,
    verbose = True,
)

print('\n✅ Entrenamiento YOLO11 completado')


WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_combined_all.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_combined_all, nbs

In [ ]:
# ── CELDA 7 (OPCIONAL): Entrenar solo Ecuador con YOLO11 ───────

import yaml
from ultralytics import YOLO

data_ec = {
    'train': [
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
    'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
    'nc':   1,
    'names': ['license plate'],
}

with open('/content/data_ecuador.yaml', 'w') as f:
    yaml.dump(data_ec, f, default_flow_style=False)

model_ec = YOLO('yolo11n.pt')  # ← usar YOLO11
model_ec.train(
    data     = '/content/data_ecuador.yaml',
    epochs   = 100,
    imgsz    = 640,
    batch    = 32,
    name     = 'yolo11n_ecuador_combined',
    project  = '/content/drive/MyDrive/TrafficVision/runs',
    patience = 15,
    save      = True,
    plots    = True,
    device   = 0,
    amp      = True,
    cos_lr   = True,
)

print('\n✅ Entrenamiento YOLO11 Ecuador completado')


In [12]:
# ── CELDA 8: Reanudar entrenamiento interrumpido ───────────────
import glob, os

RUNS_DIR = "/content/drive/MyDrive/TrafficVision/runs"

# Buscar último checkpoint de YOLO11 (ajusta el nombre si usaste otro)
# Específicamente buscar el checkpoint para 'yolo11n_combined_all'
last_models = glob.glob(f"{RUNS_DIR}/yolo11n_combined_all/weights/last.pt", recursive=True)

if last_models:
    last_pt = last_models[0]
    print(f"✅ Checkpoint encontrado: {last_pt}")
    size = os.path.getsize(last_pt) / (1024*1024)
    print(f"   Tamaño: {size:.1f} MB")

    model = YOLO(last_pt)
    model.train(
        data        = "/content/data_combined_all.yaml",
        epochs      = 100,
        imgsz       = 640,
        batch       = 16,          # reducir si hay problemas de memoria
        name        = "yolo11n_combined_all", # Asegúrate de que este nombre coincide con la carpeta del checkpoint si estás reanudando
        project     = f"{RUNS_DIR}",
        patience    = 15,
        save        = True,
        save_period = 5,
        plots       = True,
        device      = 0,
        amp         = True,
        resume      = True,
        cache       = 'disk',
    )
else:
    print("❌ No se encontró checkpoint para 'yolo11n_combined_all'. Ejecuta Celda 6 primero o verifica el nombre del run.")


❌ No se encontró checkpoint para 'yolo11n_combined_all'. Ejecuta Celda 6 primero o verifica el nombre del run.


In [ ]:

# ── CELDA 9: Evaluar métricas finales del modelo YOLO11 ────────
import glob

runs_dir = '/content/drive/MyDrive/TrafficVision/runs'
# Buscar modelos YOLO11
best_models = glob.glob(f'{runs_dir}/**/weights/best.pt', recursive=True)
best_models = [m for m in best_models if 'yolo11' in m]

print('Modelos entrenados con YOLO11:')
for m in best_models:
    size = os.path.getsize(m) / (1024*1024)
    print(f'  ✅ {m.split("/")[-3]} — {size:.1f} MB')

if best_models:
    # Elegir el mejor según métricas (normalmente el último)
    model_eval = YOLO(best_models[-1])
    metrics = model_eval.val(
        data   = '/content/data_combined_all.yaml',
        imgsz  = 640,
        device = 0,
        batch  = 32,
    )
    print(f'\n── Métricas YOLO11 ───────────────────────')
    print(f'  mAP@50:    {metrics.box.map50:.4f}')
    print(f'  mAP@50-95: {metrics.box.map:.4f}')
    print(f'  Precisión: {metrics.box.mp:.4f}')
    print(f'  Recall:    {metrics.box.mr:.4f}')
else:
    print("⚠️ No se encontraron modelos YOLO11 para evaluar.")